# Analyzing AFM scans in Python

This notebook runs the analysis of AFM Nanofiber Analyzer from Python, without
the GUIs: it analyzes an AFM height scan into a `.b2z` bundle, draws the
calibrated height image with the skeleton and kinks, measures every fiber into
a table, saves the table as CSV, and finally analyzes and measures several
scans together.

The GUIs and `cli.py` call the same functions (`afm_nanofiber_analyzer.pipeline`
and `afm_nanofiber_analyzer.measure`), so the same input and settings give the
same numbers here.

## Setup

The package is not on PyPI. Install it from a checkout of the repository, then
install Jupyter:

```
python -m pip install .
python -m pip install jupyterlab
```

`pip install .` installs the analysis package `afm_nanofiber_analyzer` only; the
GUIs and `cli.py` are run from the checkout.

## Inputs

The paths below are relative to the folder of this notebook,
`docs/manual/notebooks/` of a checkout, and point at the scans bundled in
`testdata/`. Replace them with your own files.

The scan size gives the lengths their unit. For a Shimadzu text export, `None`
reads it from the header (`SizeX` / `SizeY`). A file whose header does not record
it needs the size given here; otherwise the bundle has no scan size and
`measure_bundle` stops with an error.

In [ ]:
from pathlib import Path

# Input scans with their scan size (x_um, y_um). None reads the size from the
# file header; a text export whose header has none needs it given here.
DATA_DIR = Path("../../../testdata")
INPUTS = [
    (DATA_DIR / "TunicateACTOCCNF.txt", (5.0, 5.0)),
    (DATA_DIR / "_20250318-164122_T.ssp .txt", None),
]

# Folder the bundles and CSV files are written to.
OUTPUT_DIR = Path("afm_output")
OUTPUT_DIR.mkdir(exist_ok=True)

## Imports

In [ ]:
import dataclasses

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from afm_nanofiber_analyzer.blosc2_io import load_bundle, load_bundle_meta
from afm_nanofiber_analyzer.bundle_schema import pixel_size_nm, scan_size_um_from_meta
from afm_nanofiber_analyzer.measure import collect_fiber_stats, measure_bundle, write_fiber_csv
from afm_nanofiber_analyzer.pipeline import ProcParams, process_file

## Analysis settings

`ProcParams` holds every analysis setting, and the defaults are those of GUI01.
Change a setting by passing it, for example `ProcParams(bg_method="tophat")`.
The settings are written beside the bundle as `<input_stem>_param.json`; what
each one does is explained in [`docs/algorithms.md`](../../algorithms.md).

In [ ]:
params = ProcParams()
dataclasses.asdict(params)

## Analyzing one scan

`process_file` runs background calibration, binarization, skeletonization and
kink detection, and writes `<input_stem>.b2z` and `<input_stem>_param.json` to
`OUTPUT_DIR`. The two warning lists name reasons the background calibration or
the pixel noise may have affected the fiber heights; they are empty when none
applies.

In [ ]:
input_path, scan_size_um = INPUTS[0]
result = process_file(
    str(input_path), params, output_dir=str(OUTPUT_DIR), scan_size_um=scan_size_um,
)
print("bundle:", result.bundle_path)
print("parameters:", result.param_path)
print(f"elapsed: {result.elapsed_s:.1f} s")
print("background warnings:", result.background_warnings)
print("noise warnings:", result.noise_warnings)

## Measuring the fibers

`measure_bundle` traces every fiber of the bundle and returns one `FiberStats`
per fiber: length, height, width, kink count and angles, and more. How each
value is computed is explained in
[`docs/gui04_measurements.md`](../../gui04_measurements.md).

Called like this, it measures the bundle as analyzed: no fiber is excluded and
no fragments are connected.

In [ ]:
measured = measure_bundle(result.bundle_path)
fibers = pd.DataFrame([dataclasses.asdict(s) for s in measured.stats])
print(f"{len(fibers)} fibers, scan size {measured.scale_um} x {measured.scale_y_um} µm "
      f"({measured.scale_source})")
fibers.head()

## Looking at the result

The arrays of a bundle are read with `load_bundle`; `calibrated` is the
background-corrected height in nm. The pixel size comes from the scan size
recorded in the bundle through `pixel_size_nm`, so the axes are in µm.

On the right, each measured fiber is drawn along the centerline its values were
measured on (`Fiber.xtrack`, `Fiber.ytrack`), with its kinks
(`Fiber.kink_indices`), placed on the image as GUI04 places them.

In [ ]:
height = load_bundle(result.bundle_path, keys=["calibrated"])["calibrated"]
scan_x_um, scan_y_um = scan_size_um_from_meta(load_bundle_meta(result.bundle_path))
rows, cols = height.shape
px_x_um = pixel_size_nm(scan_x_um, cols) / 1000
px_y_um = pixel_size_nm(scan_y_um, rows) / 1000
extent = (0, cols * px_x_um, rows * px_y_um, 0)
vmin, vmax = np.percentile(height, [1, 99])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5.5), constrained_layout=True)
im = ax1.imshow(height, cmap="afmhot", vmin=vmin, vmax=vmax, extent=extent)
ax1.set_title("Calibrated height")
fig.colorbar(im, ax=ax1, label="Height (nm)")

ax2.imshow(height, cmap="gray", vmin=vmin, vmax=vmax, extent=extent)
for i, fiber in enumerate(measured.fibers):
    # The centerline is in pixels from the corner of the fiber's bounding box,
    # which fiber.data[0] (x) and fiber.data[1] (y) give.
    x = (fiber.xtrack + fiber.data[0] + 0.5) * px_x_um
    y = (fiber.ytrack + fiber.data[1] + 0.5) * px_y_um
    ax2.plot(x, y, color="deepskyblue", linewidth=1.2,
             label="Centerline" if i == 0 else None)
    kinks = np.asarray(fiber.kink_indices, dtype=int)
    ax2.scatter(x[kinks], y[kinks], s=60,
                facecolors="none", edgecolors="orange", linewidths=1.5,
                label="Kink" if i == 0 else None)
ax2.set_title("Measured centerlines and kinks")
ax2.legend(loc="upper right")
for ax in (ax1, ax2):
    ax.set_xlabel("x (µm)")
    ax.set_ylabel("y (µm)")
plt.show()

## Saving the table

`write_fiber_csv` writes the same columns as `cli.py measure`. The file is UTF-8
with a BOM, so Excel on Japanese Windows opens it without garbled text.

In [ ]:
csv_path = OUTPUT_DIR / (Path(result.bundle_path).stem + "_fibers.csv")
write_fiber_csv(
    str(csv_path), measured.stats, scale_um=measured.scale_um,
    scale_y_um=measured.scale_y_um, scale_source=measured.scale_source,
)
print("written:", csv_path)

## Distributions

A fiber that reaches the edge of the scan (`touches_frame`) or whose end is cut
at a crossing (`cut_end_count`) continues beyond what was measured, so its
`length_nm` is only a lower bound. The length histogram below stacks those
fibers separately. GUI03 offers corrections for such fibers; see
[`docs/gui03_statistics.md`](../../gui03_statistics.md).

In [ ]:
lower_bound = fibers["touches_frame"].astype(bool) | (fibers["cut_end_count"] > 0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
ax1.hist([fibers.loc[~lower_bound, "length_nm"], fibers.loc[lower_bound, "length_nm"]],
         bins=20, stacked=True, label=["Both ends are fiber ends", "Length is a lower bound"])
ax1.set_xlabel("Length (nm)")
ax1.set_ylabel("Fibers")
ax1.legend()
ax2.hist(fibers["height_median_nm"].dropna(), bins=20)
ax2.set_xlabel("Median height (nm)")
ax2.set_ylabel("Fibers")
plt.show()

## Several scans

`collect_fiber_stats` measures a list of bundles and returns the statistics per
bundle, together with the bundles it could not measure and why.

With `apply_exclusions=True` and `apply_connection=True` it applies what GUI04
saved beside a bundle: the excluded fibers (`<stem>_excluded.json`) and the
connected fragments (`<stem>_connect.json`). A bundle without these files is
measured as analyzed, as it is here. A connection file recorded against a
different skeleton, because the bundle was analyzed again after it was saved, is
refused and listed in `errors`.

In [ ]:
bundle_paths = [result.bundle_path]
for input_path, scan_size_um in INPUTS[1:]:
    bundle_paths.append(process_file(
        str(input_path), params, output_dir=str(OUTPUT_DIR), scan_size_um=scan_size_um,
    ).bundle_path)

per_bundle, errors = collect_fiber_stats(
    bundle_paths, apply_exclusions=True, apply_connection=True,
)
for path, message in errors:
    print("failed:", path, message)

all_fibers = pd.concat(
    [pd.DataFrame([dataclasses.asdict(s) for s in stats]).assign(bundle=Path(path).name)
     for path, stats in per_bundle],
    ignore_index=True,
)
all_fibers.groupby("bundle")[["length_nm", "height_median_nm"]].agg(["count", "median"])